# Scraping with GCD rerun: rotated metric

The previous dual current is projected onto both global constraints of the target source in the best-conditioned rotated real-power metric $\mathbf{W}^\mathrm{H}\mathbf{H}_{\theta}\mathbf{W}$, $\mathbf{H}_\theta = \cos\theta\,\mathbf{R} - \sin\theta\,\mathbf{X}$ (`metricStep` with `Hth(thOpt)` in `scrapingCommon.py`). The fixed-projector version of the same step is in `scrapingBothEllipsoids.ipynb`.

**One scraping step with the target on both global constraints** (`main.tex`, Section *Scraping*, paragraph *Target on both global constraints*):
1. **Dual solution.** Solve the dual for the current source $V_j$, warm started from $\lambda_{j-1}$. This gives $\lambda_j$, the Lagrangian $A_{\phi j}$, the bound $D_j$ and the dual current $x_{\phi j}$.
2. **Projection** (`projectBoth` in `scrapingCommon.py`). Compute $x_{\mathrm{p},j} = \arg\min_x (x - x_{\phi j})^H M (x - x_{\phi j})$ subject to the global real and reactive power of the target source $V_\mathrm{t}$.
   This is a QCQP of its own in Dolphindes form ($A_0 = M$, $s_0 = M x_{\phi j}$, constraints with $s_1 = -V_\mathrm{t}/2$ and $P = \mathbf{1}, -j\mathbf{1}$), solved by its two-multiplier dual (Newton from $\lambda = 0$).
   The objective is strictly concave and there are only two complex constraints, so the dual is tight and the result is the exact projection.
   - **Rotated metric:** $M = W^H H_{\theta_\mathrm{opt}} W / \|\cdot\|_2$, with $H_\theta = \cos\theta R - \sin\theta X$ at the best-conditioned angle ($0.0277°$; condition number $4.9\cdot10^4$ instead of $1.2\cdot10^7$ at $\theta = 0$).
   - **$A_0$ metric:** the real-power ellipsoid metric $\theta = 0$.
3. **Linear-term change.** $\delta s_j = A_{\phi j}(x_{\mathrm{p},j} - x_{\phi j})$. The near-kernel part of the move is annihilated.
4. **Source change.** $\delta V_j = S(\lambda_j)^{-1}\delta s_j = -2\,\mathrm{diag}(q_j)^{-1} Z_0^H \delta s_j$, where $q = \sum_k \lambda_k p_k$ is the multiplier-weighted projector diagonal (`mult` in `scrapingCommon.py`), since $s(\lambda; V) = -\tfrac12 W^H \mathrm{diag}(q) V$.
5. **Step size.** $V_{j+1} = V_j + \eta_j \delta V_j$, starting from $\eta_j = 1$. The dual is re-solved with fixed projectors from $\lambda_j$, and $\eta_j$ is halved until the dual value matches its first-order prediction $D_j + 2\,\mathrm{Re}[x_{\phi j}^H \eta_j \delta s_j]$ within $\tau |D_j|$, $\tau = 10^{-2}$.

At fixed multipliers, one full step would put the stationary point on the target up to its near-kernel part. The re-solved multipliers change $A_{\phi j}$ and let the near-kernel component drift, so the step is repeated with a new projection each time.

6. **GCD continuation.** After the accepted step, GCD is continued on $V_{j+1}$ for `NGCD_RE` iterations. The projectors of the previous run are kept, and the projector number is capped by merging.

**Settings.** The target source is the own one, $V_\mathrm{t} = V_j$, by default (`TARGET = 'own'`; `'orig'` targets the original $V$).

**Stopping.** Scraping stops at the first of:
- the bound falling below 70 % of $D_0$;
- the perturbed problem becoming strongly dual ($|d(\mathbf{V}_j)|$ and $|c_\mathrm{im}(\mathbf{V}_j)|/r^2$ below `STRONG_TOL`);
- no $\eta \geq 10^{-8}$ passing the linearity check;
- `JMAX` steps.

The initial GCD state comes from `scrapingInitGCD.py`.

In [ ]:
import os
STRATEGY = 'rotated'
NGCD_RE = int(os.environ.get('NGCD_RE', 10))      # GCD iterations per scraping step
JMAX = int(os.environ.get('JMAX', 50))            # maximum number of scraping steps
FIT = int(os.environ.get('FIT', 1))               # chained density fit
TARGET = os.environ.get('TARGET', 'own')          # constraints of the own source V_j or of the original V
ETA0 = float(os.environ.get('ETA0', 1.0))
STRONG_TOL = float(os.environ.get('STRONG_TOL', 1e-4))
INIT = os.environ.get('INIT')                     # results/scrapingInitGCD_*.npz, default newest
TAG = os.environ.get('TAG', 'local')
from scrapingCommon import *
Q0, ngcd0 = loadInitialGCD(INIT)
ngcdMax = Q0.n_proj_constr if ngcd0 is None else 2 + 2*ngcd0
print(f'{STRATEGY}: NGCD_RE {NGCD_RE}, JMAX {JMAX}, TARGET {TARGET}, ETA0 {ETA0}, STRONG_TOL {STRONG_TOL}, max projectors {ngcdMax}')

In [ ]:
step = metricStep(lambda A: Msym(WH @ Hth(thOpt) @ W), TARGET)

In [ ]:
t0 = time.time()
hist = scrapeGCD(Q0, step, NGCD_RE, JMAX, ngcdMax, eta0=ETA0, strongTol=STRONG_TOL)
print(f"{len(hist) - 1} steps, {time.time() - t0:.0f}s, stop: {hist[-1]['stop'] or 'JMAX reached'}")

In [ ]:
%matplotlib inline
fig = plotHistory(hist, f'scraping with GCD rerun: {STRATEGY}', f'results/scrapingGCD_{STRATEGY}_{TAG}.png')

In [ ]:
D0 = hist[0]['D']
print(f"{'j':>3s} {'eta':>8s} {'D/D0':>8s} {'e':>9s} {'d(V)':>10s} {'d(Vj)':>10s} {'cim(V)':>10s} {'cim(Vj)':>10s} {'viol':>9s}")
for h in hist:
    print(f"{h['j']:3d} {h['eta']:8.2e} {h['D']/D0:8.5f} {h['e']:9.2e} {h['d']:+10.2e} {h['dOwn']:+10.2e} {h['cim']:+10.2e} {h['cimOwn']:+10.2e} {h['viol']:9.2e}")

In [ ]:
fitOut = chainedFit(hist) if FIT else None

In [ ]:
saveResults(f'results/scrapingGCD_{STRATEGY}_{TAG}.npz', hist, fitOut, D0=D0, Pfull=Pfull, strategy=STRATEGY,
            target=TARGET, ngcdRe=NGCD_RE, init=str(INIT))